# 1. FP-Growth — Introduction & Intuition

## What is FP-Growth?

**FP-Growth (Frequent Pattern Growth)** is an algorithm for finding **frequent itemsets** in transactional data.

Its purpose is the same as Apriori:

$$
\text{Transactions}
\rightarrow
\text{Frequent Itemsets}
\rightarrow
\text{Association Rules}
$$

But it solves Apriori's main problem:

$$
\text{Too many candidates}
+
\text{Repeated database scans}
$$

---

## 1.1 Core Idea

Instead of generating candidate itemsets like Apriori, FP-Growth:

$$
\boxed{
\text{Compress Transactions}
\rightarrow
\text{Build FP-Tree}
\rightarrow
\text{Mine the Tree}
}
$$

The **FP-Tree** stores transactions in a compressed tree structure.

Transactions with common items can share the same path.

### Example

Suppose:

```text
T1: A B C
T2: A B
T3: A C
```

T1 and T2 both start with:

```text
A → B
```

Instead of storing `A → B` twice, the FP-Tree can share that path and store **counts** on nodes.

Conceptually:

```text
       Root
        |
        A
       / \
      B   C
      |
      C
```

The numbers/counts attached to nodes represent how many transactions pass through that node.

---

## 1.2 Why is it called "Frequent Pattern Growth"?

Because it starts with frequent items and **grows patterns** from them.

Instead of:

$$
\text{Generate candidates}
\rightarrow
\text{test candidates}
$$

it does:

$$
\text{Find frequent items}
\rightarrow
\text{Build compact structure}
\rightarrow
\text{Grow frequent patterns}
$$

---

## 1.3 FP-Growth vs Apriori — Intuition

### Apriori

Imagine trying combinations:

```text
A
B
C
↓
AB
AC
BC
↓
ABC
...
```

It explicitly creates candidates and checks them.

### FP-Growth

Instead, it says:

> "Let me compress the transaction data first, then discover the patterns directly from that compressed representation."

So:

$$
\boxed{
\text{Apriori → Candidate-based}
}
$$

$$
\boxed{
\text{FP-Growth → Tree-based}
}
$$

---

## 1.4 High-Level FP-Growth Workflow

There are two major phases.

### Phase 1 — Construct FP-Tree

$$
\text{Transactions}
\rightarrow
\text{Count frequencies}
\rightarrow
\text{Remove infrequent items}
\rightarrow
\text{Order items}
\rightarrow
\text{Build FP-Tree}
$$

### Phase 2 — Mine FP-Tree

$$
\text{FP-Tree}
\rightarrow
\text{Conditional Pattern Base}
\rightarrow
\text{Conditional FP-Tree}
\rightarrow
\text{Frequent Itemsets}
$$

Then:

$$
\text{Frequent Itemsets}
\rightarrow
\text{Association Rules}
$$

---

## 1.5 The Most Important Difference

Apriori:

$$
\boxed{
C_k \rightarrow \text{Prune} \rightarrow L_k
}
$$

FP-Growth:

$$
\boxed{
\text{FP-Tree} \rightarrow \text{Pattern Mining}
}
$$

**FP-Growth does not need explicit candidate generation**, which is the main reason it can be much more efficient than Apriori on large transaction datasets.


# `03_FP_Growth.ipynb`

# 2. Step 1 — Count Item Frequencies

Before building the FP-Tree, FP-Growth first scans the transactions **once** to determine how frequently each item occurs.

Consider:

| Transaction | Items   |
| ----------- | ------- |
| T1          | A, B, C |
| T2          | A, B    |
| T3          | A, C    |
| T4          | B, C    |
| T5          | A, B, C |

Assume:

$$
\text{Minimum Support}=40\%
$$

There are **5 transactions**, so the minimum support count is:

$$
5\times0.40=2
$$

Therefore, an item must occur in **at least 2 transactions**.

### Count each item

| Item | Count | Support |
| ---- | ----: | ------: |
| A    |     4 |     80% |
| B    |     4 |     80% |
| C    |     4 |     80% |

All three are frequent.

So:

$$
L_1=\{A,B,C\}
$$

---

# 3. Step 2 — Remove Infrequent Items

Now remove any item whose support is below minimum support.

In our example:

```text
A → 4 ✓
B → 4 ✓
C → 4 ✓
```

Therefore, nothing is removed.

If we had:

```text
D → 1
```

then:

$$
1 < 2
$$

so `D` would be removed from **every transaction** before constructing the tree.

This is important because FP-Growth only needs to work with **frequent items**.

---

# 4. Step 3 — Order the Items

This step is very important.

FP-Growth sorts items inside **each transaction according to their global frequency**, usually from highest to lowest.

Suppose the frequency is:

$$
A=4,\quad B=4,\quad C=3
$$

Then we could choose the order:

$$
A\rightarrow B\rightarrow C
$$

Every transaction is reordered using this same global order.

For example:

```text
Original:
T1: C A B

Ordered:
T1: A B C
```

Another:

```text
Original:
T2: B C

Ordered:
T2: B C
```

### Why do this?

Because putting common items first causes different transactions to share **longer prefixes**.

For example:

```text
T1: A B C
T2: A B
T3: A C
```

Both T1 and T2 can share:

```text
A → B
```

This is what allows the FP-Tree to **compress the transaction database**.

---

### The key idea so far

```text
Transactions
      ↓
Count frequencies
      ↓
Remove infrequent items
      ↓
Sort remaining items
      ↓
Build FP-Tree
```

The **next step is the important part: actually constructing the FP-Tree from these ordered transactions.**


# `03_FP_Growth.ipynb`

# 4. Step 4 — Building the FP-Tree

Now we actually construct the **FP-Tree**.

We'll continue with:

| Transaction | Ordered Items |
| ----------- | ------------- |
| T1          | A → B → C     |
| T2          | A → B         |
| T3          | A → C         |
| T4          | B → C         |
| T5          | A → B → C     |

All items are frequent.

---

## 4.1 Start with the Root

Initially, the tree contains only:

```text id="8a1c1p"
Root
```

The root does **not represent an item**. It is simply the starting point.

---

## 4.2 Insert T1

T1:

$$
A\rightarrow B\rightarrow C
$$

Create the path:

```text id="x8z4x7"
Root
  |
 A:1
  |
 B:1
  |
 C:1
```

The `1` means one transaction currently passes through that node.

---

## 4.3 Insert T2

T2:

$$
A\rightarrow B
$$

The path `A → B` already exists.

So we **don't create new nodes**.

Instead, increase their counts:

```text id="6y1j1h"
Root
  |
 A:2
  |
 B:2
  |
 C:1
```

Notice:

* A count became 2
* B count became 2
* C stays 1 because T2 doesn't contain C

---

## 4.4 Insert T3

T3:

$$
A\rightarrow C
$$

We already have `A`, so use the existing A node.

But after A, there is currently only B.

We therefore create a new C branch:

```text id="4f6h2n"
        Root
          |
         A:3
        /   \
      B:2   C:1
       |
      C:1
```

---

## 4.5 Insert T4

T4:

$$
B\rightarrow C
$$

This transaction starts with B, not A.

So we need another branch directly from Root:

```text id="r2m8qk"
           Root
          /    \
       A:3     B:1
      /   \      \
    B:2   C:1    C:1
     |
    C:1
```

---

## 4.6 Insert T5

T5:

$$
A\rightarrow B\rightarrow C
$$

The entire path already exists.

So we simply increase the counts:

```text id="b7m0kp"
           Root
          /    \
       A:4     B:1
      /   \      \
    B:3   C:1    C:1
     |
    C:2
```

This is our final FP-Tree.

---

# 4.7 What Do the Counts Mean?

Take this path:

```text id="3n6gqv"
Root
 |
A:4
 |
B:3
 |
C:2
```

This means:

* **A:4** → 4 transactions contain A
* **B:3 under A** → 3 transactions contain the prefix A → B
* **C:2 under B** → 2 transactions contain the prefix A → B → C

So the tree stores **frequency information directly in its nodes**.

---

# 4.8 Why is This Compression Useful?

Our original transactions were:

```text id="c9f5w1"
A B C
A B
A C
B C
A B C
```

FP-Tree compresses their common parts:

```text id="9z8p3k"
             Root
            /    \
         A:4     B:1
        /   \      \
      B:3   C:1    C:1
       |
      C:2
```

Instead of storing every transaction independently, **common prefixes are shared**.

That's the central idea behind FP-Growth:

$$
\boxed{
\text{Common transaction patterns}
\rightarrow
\text{Shared tree paths}
}
$$

The tree is now built. The next part is **how FP-Growth mines this tree to actually discover frequent itemsets**.


# 5. Header Table — Detailed Understanding

The **Header Table** is an important part of the FP-Tree. Its job is to provide a quick way to locate **every occurrence of an item inside the tree**.

Think of it as an **index for the FP-Tree**.

---

## 5.1 First, Our Transactions

We'll use the same example:

| Transaction | Ordered Items |
| ----------- | ------------- |
| T1          | A → B → C     |
| T2          | A → B         |
| T3          | A → C         |
| T4          | B → C         |
| T5          | A → B → C     |

Item frequencies:

| Item | Frequency |
| ---- | --------: |
| A    |         4 |
| B    |         4 |
| C    |         4 |

Our FP-Tree is:

```text
              Root
             /    \
           A:4    B:1
          /   \      \
        B:3   C:1    C:1
         |
        C:2
```

---

# 5.2 What Problem Does the Header Table Solve?

Look at item **C**.

There are three different C nodes:

```text
              Root
             /    \
           A:4    B:1
          /   \      \
        B:3   C:1    C:1
         |
        C:2
```

If FP-Growth wants to investigate **C**, it needs to find:

```text
C:2
C:1
C:1
```

Searching the entire tree every time would be inefficient.

So the Header Table keeps track of them.

---

# 5.3 Basic Header Table

Conceptually:

| Item | Support | Node Link |
| ---- | ------: | --------- |
| A    |       4 | → A:4     |
| B    |       4 | → B:3     |
| C    |       4 | → C:2     |

The **Node Link** for C then connects all C nodes:

```text
Header[C]
    |
    ↓
  C:2
    |
    ↓
  C:1
    |
    ↓
  C:1
```

So the header table doesn't necessarily contain every node itself.

It points to the **first occurrence**, and the node links connect the remaining occurrences.

---

# 5.4 Let's Understand One Item: C

Our tree:

```text
              Root
             /    \
           A:4    B:1
          /   \      \
        B:3   C:1    C:1
         |
        C:2
```

There are three C nodes:

### C node 1

```text
A → B → C:2
```

### C node 2

```text
A → C:1
```

### C node 3

```text
B → C:1
```

The Header Table connects these:

```text
Header Table

C
↓
C:2
↓
C:1
↓
C:1
```

Now FP-Growth can immediately locate every C occurrence.

---

# 5.5 What About B?

B occurs at two nodes:

```text
              Root
             /    \
           A:4    B:1
          /
        B:3
```

So:

```text
Header[B]
    |
    ↓
  B:3
    |
    ↓
  B:1
```

These two B nodes are connected through **B's node-link chain**.

---

# 5.6 What About A?

A occurs only once in our tree:

```text
Root
 |
A:4
```

Therefore:

```text
Header[A]
    |
    ↓
  A:4
```

There is no second A node to link to.

---

# 5.7 Complete Picture

Putting everything together:

```text
                  HEADER TABLE

              Item    Support
               A        4
               B        4
               C        4


                 FP-TREE

                    Root
                   /    \
                A:4     B:1
               /   \      \
             B:3   C:1    C:1
              |
             C:2
```

Node links conceptually:

```text
A ─────────────→ A:4

B ─────────────→ B:3 ─────────→ B:1

C ─────────────→ C:2 ─────────→ C:1 ─────────→ C:1
```

---

# 5.8 Two Different Types of Links

This distinction is important.

### Tree Links

These represent the actual parent-child structure:

```text
A
|
B
|
C
```

They tell us:

> **What is the transaction prefix/path?**

### Node Links

These connect nodes containing the **same item**:

```text
C:2 → C:1 → C:1
```

They tell us:

> **Where does this same item occur elsewhere in the tree?**

So:

$$
\boxed{
\text{Tree links}
\rightarrow
\text{Structure}
}
$$

$$
\boxed{
\text{Node links}
\rightarrow
\text{Same-item occurrences}
}
$$

---

# 5.9 Why Is the Header Table Important for Mining?

Suppose FP-Growth wants to mine **C**.

It does:

```text
Header[C]
     ↓
Find all C nodes
     ↓
Trace their prefix paths
     ↓
Build Conditional Pattern Base
```

For C, it finds:

```text
C:2 ← A → B → C
C:1 ← A → C
C:1 ← B → C
```

The prefixes **before C** are:

```text
A → B
A
B
```

These prefix paths will be used to construct the **Conditional Pattern Base for C**.

That is the next major step in FP-Growth.

### The one thing to remember

> **Header Table = an index that tells FP-Growth where every item occurs in the FP-Tree, using node links to connect the same items.**


# 6. Step 6 — Conditional Pattern Base

Now we use the **Header Table** to start mining the FP-Tree.

The first important concept is the **Conditional Pattern Base (CPB)**.

---

## 6.1 What is a Conditional Pattern Base?

A **Conditional Pattern Base** is the collection of **prefix paths leading to a particular item**, along with their counts.

In simple words:

> Pick an item → find every occurrence of it → look at what items came before it.

We will use **C**.

---

## 6.2 Find All C Nodes

From our FP-Tree:

```text
              Root
             /    \
           A:4    B:1
          /   \      \
        B:3   C:1    C:1
         |
        C:2
```

Header Table tells us where all C nodes are:

```text
C
↓
C:2
↓
C:1
↓
C:1
```

Now we examine each C node.

---

## 6.3 C Node 1 — C:2

Path to this node:

```text
Root → A:4 → B:3 → C:2
```

The **prefix before C** is:

```text
A → B
```

The C node has count **2**, so this path contributes:

```text
[A, B] : 2
```

---

## 6.4 C Node 2 — C:1

Path:

```text
Root → A:4 → C:1
```

The prefix before C is:

```text
A
```

Count = 1.

So:

```text
[A] : 1
```

---

## 6.5 C Node 3 — C:1

Path:

```text
Root → B:1 → C:1
```

The prefix before C is:

```text
B
```

Count = 1.

So:

```text
[B] : 1
```

---

# 6.6 Conditional Pattern Base for C

Therefore:

| Prefix Path | Count |
| ----------- | ----: |
| A → B       |     2 |
| A           |     1 |
| B           |     1 |

So:

$$
\boxed{
CPB(C)=
\{(A,B):2,\ (A):1,\ (B):1\}
}
$$

Notice something important:

**C itself is not included in the prefix paths.**

We are asking:

> "What patterns occur **before C**?"

---

# 6.7 Why Do We Do This?

We are trying to discover patterns that frequently occur **together with C**.

From the CPB:

```text
A → B : 2
A     : 1
B     : 1
```

We can calculate:

### A with C

A appears in:

$$
2+1=3
$$

transactions associated with C.

So:

$$
Support(A,C)=\frac{3}{5}=60\%
$$

### B with C

B appears in:

$$
2+1=3
$$

So:

$$
Support(B,C)=\frac{3}{5}=60\%
$$

### A, B with C

The path `A → B` has count 2.

So:

$$
Support(A,B,C)=\frac{2}{5}=40\%
$$

Therefore:

$$
\{A,C\},\{B,C\},\{A,B,C\}
$$

are frequent with our 40% minimum support.

---

# 6.8 The Key Idea

The Conditional Pattern Base is essentially a **smaller transaction database focused on one item**.

For C:

```text
Original Database
       ↓
Find C
       ↓
Look at paths before C
       ↓
Conditional Pattern Base
       ↓
Find frequent combinations with C
```

So:

$$
\boxed{
\text{Conditional Pattern Base}
=
\text{Prefix paths leading to a chosen item}
}
$$


# 8. Conditional FP-Tree

We already have the **Conditional Pattern Base (CPB)** for `C`:

| Prefix Path | Count |
| ----------- | ----: |
| A → B       |     2 |
| A           |     1 |
| B           |     1 |

Now we use this CPB to build a **smaller FP-Tree specifically for C**.

---

## 8.1 Why Build a Conditional FP-Tree?

The original FP-Tree contains information about **all items**.

But if we want to find patterns associated with `C`, we don't need the entire tree.

We only care about:

> **What combinations of items frequently occur before C?**

So we create a smaller tree containing only the relevant information.

$$
\text{CPB for C}
\rightarrow
\text{Conditional FP-Tree for C}
$$

---

# 8.2 Count Items in the CPB

Our CPB:

```text
A → B : 2
A     : 1
B     : 1
```

Count how often each item appears:

### A

A appears in:

```text
A → B : 2
A     : 1
```

Therefore:

$$
Count(A)=2+1=3
$$

### B

B appears in:

```text
A → B : 2
B     : 1
```

Therefore:

$$
Count(B)=2+1=3
$$

So:

| Item | Conditional Count |
| ---- | ----------------: |
| A    |                 3 |
| B    |                 3 |

Our minimum support count is still:

$$
2
$$

Therefore both A and B remain.

---

# 8.3 Order the Items

Both have equal frequency:

$$
A=3,\quad B=3
$$

We can choose:

$$
A\rightarrow B
$$

as the ordering.

Now reorder the CPB paths accordingly:

```text
A → B : 2
A     : 1
B     : 1
```

They are already in the correct order.

---

# 8.4 Build the Conditional FP-Tree

Start with:

```text id="y9x8tk"
Root
```

### Insert `A → B : 2`

```text id="6knx0w"
Root
  |
 A:2
  |
 B:2
```

### Insert `A : 1`

A already exists:

```text id="v2x0j8"
Root
  |
 A:3
  |
 B:2
```

### Insert `B : 1`

This doesn't start with A, so create a separate B branch:

```text id="m1br8j"
       Root
      /    \
    A:3    B:1
     |
    B:2
```

This is the **Conditional FP-Tree for C**.

---

# 8.5 What Does This Tree Tell Us?

The tree represents the patterns that occur **before C**.

```text id="9bq4dn"
       Root
      /    \
    A:3    B:1
     |
    B:2
```

From this:

### A occurs with C

$$
Count(A)=3
$$

### B occurs with C

$$
Count(B)=3
$$

### A and B occur together with C

The shared path:

$$
A\rightarrow B
$$

has count 2.

Therefore:

$$
Count(A,B,C)=2
$$

Since our minimum support count is 2:

$$
\{A,C\},\{B,C\},\{A,B,C\}
$$

are frequent.

---

# 8.6 CPB vs Conditional FP-Tree

This distinction is important:

### Conditional Pattern Base

Raw collection of prefix paths:

```text id="z2x9m0"
A → B : 2
A     : 1
B     : 1
```

### Conditional FP-Tree

Compressed tree built from those paths:

```text id="7d5c1m"
       Root
      /    \
    A:3    B:1
     |
    B:2
```

So:

$$
\boxed{
CPB
\rightarrow
Conditional\ FP\text{-}Tree
}
$$

The **CPB is the input**, while the **Conditional FP-Tree is the compressed representation used for further mining**.


# 9. Mining the Conditional FP-Tree

Now we have the **Conditional FP-Tree for C**:

```text
        Root
       /    \
     A:3    B:1
      |
     B:2
```

The goal is to use this tree to find **frequent patterns associated with C**.

---

## 9.1 Start With the Selected Item

We originally selected:

$$
C
$$

So everything we discover now is being considered **with C**.

This is what the notation:

$$
\boxed{CPB(B\mid C)}
$$

means.

Read it as:

> **Conditional Pattern Base of B, given C.**

The `C` means that we are already working inside the **C-conditional tree**.

---

## 9.2 What Is a Prefix?

A **prefix** is simply:

> **The items that appear before the selected item on its path from the Root.**

Consider the conditional tree:

```text
        Root
       /    \
     A:3    B:1
      |
     B:2
```

We now select **B**.

There are two B nodes.

### First B

```text
Root
 |
 A:3
 |
 B:2
```

The selected item is `B`.

What comes before B?

$$
A
$$

Therefore:

```text
Prefix = A
```

The B node has count 2, so:

```text
A : 2
```

means:

> The prefix A occurs before this B for 2 transactions.

---

### Second B

```text
Root
 |
 B:1
```

There is nothing before this B.

Therefore, this node gives us **no prefix pattern**.

---

## 9.3 Constructing the Conditional Pattern Base

Therefore:

$$
\boxed{
CPB(B\mid C)=\{A:2\}
}
$$

Don't think of this as complicated notation.

It simply means:

> **While we are looking for patterns associated with C, B has a prefix A with count 2.**

---

# 9.4 How Does This Create a Frequent Pattern?

We found:

```text
A → B : 2
```

inside the **C-conditional tree**.

Remember: this tree was created specifically while mining **C**.

Therefore, `C` is already part of the pattern we are constructing.

So:

```text
A → B
```

combined with our selected `C` gives:

$$
\boxed{\{A,B,C\}}
$$

---

## 9.5 Verify Using the Original Transactions

Our original transactions were:

| Transaction | Items   |
| ----------- | ------- |
| T1          | A, B, C |
| T2          | A, B    |
| T3          | A, C    |
| T4          | B, C    |
| T5          | A, B, C |

Which transactions contain **A + B + C**?

* T1 → ✓
* T2 → ✗ C missing
* T3 → ✗ B missing
* T4 → ✗ A missing
* T5 → ✓

Therefore:

$$
Count(\{A,B,C\})=2
$$

Minimum support count:

$$
5\times40\%=2
$$

So:

$$
\boxed{\{A,B,C\}\text{ is frequent}}
$$

---

# 9.6 Other Patterns We Get

Because we are mining **C**, we can also get the individual combinations:

From A:

$$
\{A,C\}
$$

From B:

$$
\{B,C\}
$$

From A and B together:

$$
\{A,B,C\}
$$

Therefore, the patterns involving C include:

$$
\boxed{
\{C\},\{A,C\},\{B,C\},\{A,B,C\}
}
$$

---

# 9.7 Why Is This Called "Growth"?

We started with the selected item:

$$
C
$$

Then found items that can be added to it:

$$
C\rightarrow AC
$$

$$
C\rightarrow BC
$$

and then:

$$
C\rightarrow ABC
$$

So frequent patterns are **grown by adding items to an existing pattern**.

That's the **Pattern Growth** idea in FP-Growth.

---

# 9.8 Complete Logic

The process is:

```text
Select C
   ↓
Build C's Conditional Pattern Base
   ↓
Build C's Conditional FP-Tree
   ↓
Mine the conditional tree
   ↓
Find A → B : 2
   ↓
C was already selected
   ↓
A + B + C
   ↓
{A,B,C} is frequent
```

### Most important idea

> **When mining a conditional tree for C, C is already part of the context. Any frequent pattern found inside that tree is combined with C.**

So:

$$
\boxed{
\text{Prefix before B} = A
}
$$

$$
\boxed{
A+B+\underbrace{C}_{\text{already selected}}
\rightarrow
\{A,B,C\}
}
$$


# `03_FP_Growth.ipynb`

# 10. Complete FP-Growth Dry Run

Now let's put **everything together** using the same dataset.

## Dataset

| Transaction | Items   |
| ----------- | ------- |
| T1          | A, B, C |
| T2          | A, B    |
| T3          | A, C    |
| T4          | B, C    |
| T5          | A, B, C |

Minimum support:

$$
40\%
$$

Therefore:

$$
\text{Minimum support count}=5\times0.4=2
$$

---

## Step 1 — Count Item Frequencies

| Item | Count |
| ---- | ----: |
| A    |     4 |
| B    |     4 |
| C    |     4 |

All are frequent.

---

## Step 2 — Order Items

Since all have equal frequency, choose:

$$
A\rightarrow B\rightarrow C
$$

Reordered transactions:

```text
T1: A → B → C
T2: A → B
T3: A → C
T4: B → C
T5: A → B → C
```

---

## Step 3 — Build FP-Tree

After inserting all transactions:

```text
              Root
             /    \
           A:4    B:1
          /   \      \
        B:3   C:1    C:1
         |
        C:2
```

The counts represent how many transactions pass through each node.

---

## Step 4 — Header Table

The Header Table lets us locate all occurrences of each item.

Conceptually:

```text
A → A:4

B → B:3 → B:1

C → C:2 → C:1 → C:1
```

The node links connect nodes containing the same item.

---

# Step 5 — Select C

We now start mining from the item `C`.

Find every C node using the Header Table.

### C:2

Path:

```text
Root → A → B → C
```

Prefix before C:

```text
A → B : 2
```

### C:1

Path:

```text
Root → A → C
```

Prefix:

```text
A : 1
```

### C:1

Path:

```text
Root → B → C
```

Prefix:

```text
B : 1
```

Therefore:

$$
\boxed{
CPB(C)=
\{(A,B):2,\ (A):1,\ (B):1\}
}
$$

---

# Step 6 — Build C's Conditional FP-Tree

Count items inside C's CPB:

$$
A=3,\qquad B=3
$$

Both satisfy the minimum count of 2.

Build the conditional tree:

```text
       Root
      /    \
    A:3    B:1
     |
    B:2
```

This tree represents the patterns that occur **before C**.

---

# Step 7 — Mine C's Conditional Tree

Now select `B`.

There are two B nodes.

### B:2

```text
Root → A → B
```

Prefix:

```text
A : 2
```

### B:1

```text
Root → B
```

No prefix.

Therefore:

$$
CPB(B\mid C)=\{A:2\}
$$

Remember:

* `B` = item we're currently examining
* `C` = the item we originally conditioned on
* `A:2` = A occurs before B with count 2

Since we're already mining **C**, the pattern:

$$
A+B
$$

becomes:

$$
\boxed{\{A,B,C\}}
$$

because C is already part of the context.

---

# Step 8 — Frequent Itemsets Found

From the C branch we obtain:

$$
\{C\}
$$

$$
\{A,C\}
$$

$$
\{B,C\}
$$

$$
\{A,B,C\}
$$

We would similarly mine the other items to discover the complete set of frequent itemsets.

For this dataset, the final frequent itemsets are:

### 1-itemsets

$$
\{A\},\{B\},\{C\}
$$

### 2-itemsets

$$
\{A,B\},\{A,C\},\{B,C\}
$$

### 3-itemset

$$
\{A,B,C\}
$$

---

# The Entire FP-Growth Process

This is the flow you should remember:

```text
TRANSACTIONS
     ↓
Count item frequencies
     ↓
Remove infrequent items
     ↓
Order items by frequency
     ↓
Build FP-Tree
     ↓
Build Header Table
     ↓
Select an item
     ↓
Follow its node links
     ↓
Find prefix paths
     ↓
Conditional Pattern Base
     ↓
Build Conditional FP-Tree
     ↓
Mine the conditional tree
     ↓
Grow frequent patterns
     ↓
FREQUENT ITEMSETS
     ↓
Generate Association Rules
```

### The central idea

Apriori does:

$$
\boxed{
\text{Generate candidates}
\rightarrow
\text{Prune}
\rightarrow
\text{Count}
}
$$

FP-Growth does:

$$
\boxed{
\text{Compress}
\rightarrow
\text{Find prefix paths}
\rightarrow
\text{Build conditional trees}
\rightarrow
\text{Grow patterns}
}
$$

The **FP-Tree is the compressed database**, and the **Conditional Pattern Base + Conditional FP-Tree are what allow FP-Growth to recursively discover frequent combinations without explicitly generating all candidates.**


# `03_FP_Growth.ipynb`

# 11. FP-Growth Implementation in Python

FP-Growth can be implemented using **`mlxtend`**, just like Apriori.

The important difference is that instead of calling `apriori()`, we call **`fpgrowth()`**.

---

## 11.1 Prepare the Transaction Data

Suppose our transactions are:

```python
transactions = [
    ["A", "B", "C"],
    ["A", "B"],
    ["A", "C"],
    ["B", "C"],
    ["A", "B", "C"]
]
```

FP-Growth expects a **one-hot encoded DataFrame**.

For example:

|    |  A |  B |  C |
| -- | -: | -: | -: |
| T1 |  1 |  1 |  1 |
| T2 |  1 |  1 |  0 |
| T3 |  1 |  0 |  1 |
| T4 |  0 |  1 |  1 |
| T5 |  1 |  1 |  1 |

---

## 11.2 Apply FP-Growth

```python
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets = fpgrowth(
    df,
    min_support=0.4,
    use_colnames=True
)
```

The output contains:

| support | itemsets  |
| ------: | --------- |
|     0.8 | {A}       |
|     0.8 | {B}       |
|     0.8 | {C}       |
|     0.6 | {A, B}    |
|     0.6 | {A, C}    |
|     0.6 | {B, C}    |
|     0.4 | {A, B, C} |

These are the **frequent itemsets**.

---

## 11.3 Generate Association Rules

Just like Apriori, FP-Growth itself gives us **frequent itemsets**, not the final rules.

We can use `association_rules()`:

```python
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.7
)
```

Then we can filter using confidence and lift:

```python
rules[
    (rules["confidence"] >= 0.7) &
    (rules["lift"] > 1)
]
```

---

## 11.4 Apriori vs FP-Growth in Code

The main change is very small:

**Apriori:**

```python
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    df,
    min_support=0.4,
    use_colnames=True
)
```

**FP-Growth:**

```python
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets = fpgrowth(
    df,
    min_support=0.4,
    use_colnames=True
)
```

Everything after finding the frequent itemsets can remain essentially the same.

---

## 11.5 What the Library Hides

When we call:

```python
fpgrowth(df, min_support=0.4)
```

the library handles the internal process:

```text
Transactions
     ↓
Frequency counting
     ↓
Remove infrequent items
     ↓
Order items
     ↓
Build FP-Tree
     ↓
Header Table
     ↓
Conditional Pattern Bases
     ↓
Conditional FP-Trees
     ↓
Pattern Growth
     ↓
Frequent Itemsets
```

So when learning FP-Growth, the important thing is **understanding this internal process**, not just knowing the `fpgrowth()` function.

The implementation itself is quite short because the algorithm is handled by the library.


Yes — but there is an important distinction.

### Apriori

If you're using `mlxtend`, **Apriori also expects a one-hot encoded DataFrame**.

```python
frequent_itemsets = apriori(
    df,
    min_support=0.4,
    use_colnames=True
)
```

Here `df` should look like:

| A | B | C |
| - | - | - |
| 1 | 1 | 1 |
| 1 | 1 | 0 |
| 1 | 0 | 1 |
| 0 | 1 | 1 |
| 1 | 1 | 1 |

So you **don't give the raw transaction lists directly** to `apriori()`.

### FP-Growth

Exactly the same input format:

```python
frequent_itemsets = fpgrowth(
    df,
    min_support=0.4,
    use_colnames=True
)
```

Again:

> **FP-Growth also takes a one-hot encoded DataFrame.**

---

### So why did it look different?

Earlier, when we wrote:

```python
transactions = [
    ["A", "B", "C"],
    ["A", "B"],
    ["A", "C"],
    ["B", "C"],
    ["A", "B", "C"]
]
```

that was the **raw transaction representation**.

You need to convert it to one-hot encoding before passing it to either algorithm.

For example:

```python
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
encoded = te.fit(transactions).transform(transactions)

df = pd.DataFrame(encoded, columns=te.columns_)
```

Then:

```python
apriori(df, min_support=0.4, use_colnames=True)
```

or:

```python
fpgrowth(df, min_support=0.4, use_colnames=True)
```

### Bottom line

$$
\boxed{
\text{Raw Transactions}
\rightarrow
\text{One-Hot Encoding}
\rightarrow
\begin{cases}
\text{Apriori}\\
\text{FP-Growth}
\end{cases}
}
$$

**Both use the same input format in `mlxtend`.** The difference is what happens **internally** after the DataFrame is given to them.


# `03_FP_Growth.ipynb`

# 12. FP-Growth vs Apriori

Now that we understand both algorithms, the main comparison is straightforward.

| Feature              | Apriori              | FP-Growth                        |
| -------------------- | -------------------- | -------------------------------- |
| Main approach        | Candidate generation | Pattern growth                   |
| Data structure       | Candidate itemsets   | FP-Tree                          |
| Candidate generation | Yes                  | No explicit candidate generation |
| Pruning              | Yes                  | Not in the Apriori sense         |
| Database scans       | Multiple             | Generally fewer                  |
| Speed                | Usually slower       | Usually faster                   |
| Memory issue         | Many candidates      | FP-Tree can be large             |
| Concept              | Easier               | More complex                     |
| Large datasets       | Less suitable        | Generally better                 |

### Core Difference

**Apriori:**

$$
\text{Generate candidates}
\rightarrow
\text{Prune}
\rightarrow
\text{Count support}
\rightarrow
\text{Repeat}
$$

**FP-Growth:**

$$
\text{Build FP-Tree}
\rightarrow
\text{Conditional Pattern Base}
\rightarrow
\text{Conditional FP-Tree}
\rightarrow
\text{Grow patterns}
$$

### When to use which?

**Apriori:**

* Small datasets
* Learning/understanding association mining
* When simplicity is more important

**FP-Growth:**

* Large transaction datasets
* Many possible item combinations
* When performance matters

### One-line memory trick

$$
\boxed{\text{Apriori = Candidate Generation}}
$$

$$
\boxed{\text{FP-Growth = Pattern Growth}}
$$

---

# FP-Growth — Final Mental Model

```text
Transactions
     ↓
Count frequencies
     ↓
Remove infrequent items
     ↓
Order items
     ↓
Build FP-Tree
     ↓
Header Table
     ↓
Select an item
     ↓
Find its prefix paths
     ↓
Conditional Pattern Base
     ↓
Conditional FP-Tree
     ↓
Mine & grow patterns
     ↓
Frequent Itemsets
     ↓
Association Rules
```

The **three most important concepts to remember** are:

1. **FP-Tree** → compressed representation of transactions
2. **Header Table** → finds all occurrences of an item
3. **Conditional Pattern Base → Conditional FP-Tree** → used to recursively discover frequent patterns
